# DATA 266 Lab 2 - Part 1: Multimodal RAG over a read-only knowledge base
Modular pipeline in `rag/` (ingest -> figure descriptions -> hybrid index -> grounded answers). Run all cells top to bottom; `submission.csv` is written by Cell 7.

In [1]:
# Cell 1 - run information (required)
import subprocess, sys, random, numpy as np, torch
SEED = 266; random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
try:
    commit = subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip()
except Exception:
    commit = ""
print({"commit": commit or "n/a (not a git checkout)", "python": sys.version.split()[0], "torch": torch.__version__,
       "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu", "seed": SEED})

{'commit': 'n/a (not a git checkout)', 'python': '3.13.16', 'torch': '2.11.0+cu130', 'gpu': 'NVIDIA A100-SXM4-40GB', 'seed': 266}


In [2]:
# Cell 2 - install dependencies (Colab/Kaggle). torch/numpy/pandas are already present.
# Versions are pinned in requirements.txt; on Kaggle, turn "Internet" ON for the install, or attach the packages as a dataset.
!pip -q install -U bitsandbytes accelerate sentence-transformers chromadb pdfplumber "transformers>=4.49"
# bitsandbytes must be recent on Colab (CUDA 13). If this cell upgrades it: Runtime > Restart session, then run Cell 1 onward.

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.1 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.1 which is incompatible.
gradio 6.30.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.


In [3]:
# Cell 3 - make the `rag` package importable (works on Kaggle and Colab, zip or folder, any location).
import glob, os, pathlib, sys, zipfile

def find_repo_root():
    roots = [".", "/content", "/kaggle/working", "/kaggle/input"]
    for r in roots:                                   # 1) an already-extracted folder containing rag/config.py
        for p in glob.glob(f"{r}/**/rag/config.py", recursive=True):
            return str(pathlib.Path(p).parents[1])
    dest = "/content/_repo" if os.path.isdir("/content") else "/kaggle/working/_repo"
    for r in roots:                                   # 2) a zip that was uploaded but never extracted
        for z in glob.glob(f"{r}/**/*.zip", recursive=True):
            try:
                with zipfile.ZipFile(z) as zf:
                    if any(n.endswith("rag/config.py") for n in zf.namelist()):
                        zf.extractall(dest)
                        for p in glob.glob(f"{dest}/**/rag/config.py", recursive=True):
                            return str(pathlib.Path(p).parents[1])
            except zipfile.BadZipFile:
                pass
    return None

REPO = find_repo_root()
if REPO is None:
    raise SystemExit("rag/ not found. Upload the project folder or zip (Kaggle: Add Data > Upload; Colab: Files panel).")
sys.path.insert(0, REPO)
print("using repo at:", REPO)
# os.environ["DATA_DIR"] = "/content/data"   # only if kb_manifest.json is not found automatically
# os.environ["CAP_GPU_16GB"] = "1"           # default: on an A100, behave like the 16 GB grading GPU
# os.environ["RAG_BACKEND"] = "stub"         # ONLY to test plumbing without a GPU; answers are not real

using repo at: /content/_repo/data266-lab2-team


In [4]:
# Cell 4 - load the vision-language model (reads figures AND writes answers). 4-bit Qwen2.5-VL-7B fits a 16 GB T4.
import logging; logging.basicConfig(level=logging.INFO, format="%(name)s: %(message)s")
from rag import config
from rag.llm import load_backend
backend = load_backend()
print("backend:", backend.name)

config.json:   0%|          | 0.00/1.37k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/57.6k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

...

tokenizer_config.json:   0%|          | 0.00/5.70k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

backend: Qwen/Qwen2.5-VL-7B-Instruct


In [5]:
# Cell 5 - Task 1.1 ingestion + indexing: text chunks, tables, figure crops (with labels), figure descriptions,
# then a ChromaDB dense index + BM25 index. Low-trust material (DOC3 Appendix C) is stored but never retrieved.
from rag.pipeline import build_system
system = build_system(backend)
system.stats

/usr/local/lib/python3.13/dist-packages/transformers/models/qwen2_vl/image_processing_qwen2_vl.py:129: FutureWarning: Passing `min_pixels` and `max_pixels` to a processor call is deprecated and will be removed in v5.23. Pass in `size={'longest_edge': xxx, 'shortest_edge': xxx} to override the target size.`
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/bitsandbytes/backends/cuda/ops.py:957: UserWarning: inner dimension (3420) is not aligned for fast kernel with blocksize=64, falling back to slower implementation.
  warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

{'items': 291,
 'by_doc_kind': {'DOC1/text': 224,
  'DOC2/text': 17,
  'DOC3/text': 14,
  'DOC1/figure': 13,
  'DOC3/figure': 7,
  'DOC2/figure': 7,
  'DOC1/table': 4,
  'DOC2/table': 3,
  'DOC3/table': 2},
 'figure_descriptions': {'figures': 26,
  'described': 26,
  'failed': 0,
  'seconds': 414.5},
 'dense_enabled': True,
 'build_seconds': 435.2}

In [6]:
# Cell 6 - smoke test on the first question. Never fatal: problems are shown, not raised.
from rag.pipeline import read_questions
from rag.memory import ConversationMemory
from rag.retrieve import retrieve
from rag.answer import answer_question
import json
q = read_questions()
first = q.iloc[0]
ev = retrieve(system.index, first["question"], ConversationMemory("smoke"))
ans, dbg = answer_question(first["question"], ev, ConversationMemory("smoke"), backend)
print(json.dumps(ans, indent=1)); print("issues:", dbg["issues"]); print("evidence:", [i.item_id for i in ev.items])

{
 "answer": "The knowledge base does not contain this information.",
 "status": "not_found",
 "citations": [],
 "figure_references": []
}
issues: ['unsupported:The cited evidence does not explicitly state the specific value or fact that the', 'downgraded_not_found_unsupported']
evidence: ['DOC1-p2-c1', 'DOC1-p12-c3', 'DOC1-p2-c3', 'DOC1-p12-c4', 'DOC1-p24-c4', 'DOC1-p5-c0', 'DOC1-p1-c4', 'DOC1-p3-c3', 'DOC1-p7-Table 1-3', 'DOC1-p1-Table 1-1', 'DOC1-p2-Figure 1-1', 'DOC1-p3-Figure 1-2']


In [7]:
# Cell 7 - answer every question (conversation by conversation) and write submission.csv with no manual step.
# QUESTIONS_PATH is honoured so graders can swap in the hidden set.
import time, pathlib
from rag.pipeline import run_questions, validate_submission
from rag.config import questions_path, find_data_dir, work_dir
q = read_questions(questions_path(find_data_dir()))
t0 = time.time()
sub = run_questions(system, q, "submission.csv", work_dir() / "trace_p1.jsonl")
print(f"answered {len(sub)} questions in {time.time()-t0:.0f}s")
problems = validate_submission(sub, q)
print("format problems:", problems or "none")
sub.head(10)

/usr/local/lib/python3.13/dist-packages/transformers/models/qwen2_vl/image_processing_qwen2_vl.py:129: FutureWarning: Passing `min_pixels` and `max_pixels` to a processor call is deprecated and will be removed in v5.23. Pass in `size={'longest_edge': xxx, 'shortest_edge': xxx} to override the target size.`
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/transformers/models/qwen2_vl/image_processing_qwen2_vl.py:129: FutureWarning: Passing `min_pixels` and `max_pixels` to a processor call is deprecated and will be removed in v5.23. Pass in `size={'longest_edge': xxx, 'shortest_edge': xxx} to override the target size.`
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/transformers/models/qwen2_vl/image_processing_qwen2_vl.py:129: FutureWarning: Passing `min_pixels` and `max_pixels` to a processor call is deprecated and will be removed in v5.23. Pass in `size={'longest_edge': xxx, 'shortest_edge': xxx} to override the target size.`
  warnings.warn(
/usr/local/lib/python3.13

answered 40 questions in 327s
format problems: none


,id,conversation_id,question_id,answer,figure_references,citations,status
0,1,C001,Q001,The knowledge base does not contain this infor...,NONE,NONE,not_found
1,2,C002,Q002,not_found,NONE,NONE,not_found
2,3,C003,Q003,Real GDP is constructed as the sum of the quan...,NONE,NONE,not_found
3,4,C004,Q004,not_found,NONE,NONE,not_found
4,5,C005,Q005,The U.S. unemployment rate increased dramatica...,NONE,DOC1:p5,answered
5,6,C006,Q006,"Before the crisis, China's economy grew at an ...",Table 1-4,"DOC1:Table 1-4,DOC1:p11",answered
6,7,C007,Q007,stock markets everywhere tanked,NONE,NONE,not_found
7,8,C008,Q008,not_found,NONE,NONE,not_found
8,9,C009,Q009,not_found,NONE,NONE,not_found
9,10,C010,Q010,not_found,NONE,NONE,not_found


In [8]:
# Cell 8 - Task 1.4 evaluation on YOUR golden set (golden/golden_set.csv): baseline (dense only) vs final (hybrid).
import pandas as pd, pathlib
from rag.evaluate import evaluate
gpath = next((p for p in ["golden/golden_set.csv", "../golden/golden_set.csv"] if pathlib.Path(p).exists()), None)
golden = pd.read_csv(gpath, dtype=str, keep_default_na=False) if gpath else pd.DataFrame()
if len(golden):
    results = [evaluate(system, golden, mode=m)[0] for m in ("dense", "hybrid")]
    display(pd.DataFrame(results))
else:
    print("golden/golden_set.csv is empty - write your 25 questions first (see README).")

golden/golden_set.csv is empty - write your 25 questions first (see README).


In [9]:
# Final cell - validate submission.csv, then download it (and the trace) for the Kaggle upload
import pandas as pd
from rag.pipeline import validate_submission, read_questions
from rag.config import questions_path, find_data_dir, work_dir

q = read_questions(questions_path(find_data_dir()))
sub = pd.read_csv("submission.csv", dtype=str, keep_default_na=False)
problems = validate_submission(sub, q)
print("rows:", len(sub), "| format problems:", problems or "none")
print(sub.status.value_counts().to_string())

if problems:
    print("Fix the format problems above before uploading.")
else:
    try:
        from google.colab import files
        files.download("submission.csv")                       # -> upload this to the Kaggle competition
        files.download(str(work_dir() / "trace_p1.jsonl"))     # -> send this to me if the score is off
    except ImportError:
        print("Not on Colab: submit submission.csv from the notebook's Output tab.")

rows: 40 | format problems: none
status
answered            20
not_found           15
session_override     5


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>